In [1]:
from transformers import AutoTokenizer
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
# Встановлюємо потрібні бібліотеки; -q вимикає зайвий вивід під час встановлення
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [2]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: NVIDIA GeForce GTX 1070
Памʼять GPU: 8.6 GB


In [3]:
# Детальна інформація про GPU NVIDIA: модель, зайнята й вільна памʼять
if device == "cuda":
    !nvidia-smi

Wed Nov 11 17:44:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 582.78                 Driver Version: 582.78         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1070      WDDM  |   00000000:08:00.0  On |                  N/A |
| 71%   55C    P0             33W /  151W |    1907MiB /   8192MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) — відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** — лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) — донавчена виконувати інструкції й вести діалог.

In [ ]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

# Ваги SmolLM3 збережено в bfloat16, але T4 не підтримує його апаратно,
# тому завантажуємо у float16.
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` — інструкції для моделі (роль, стиль, обмеження);
- `user` — повідомлення користувача;
- `assistant` — відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` — для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` — додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [4]:
# Різні типи діалогів для порівняння
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    # Завершений діалог: так виглядають тренувальні дані
    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    # Діалог для інференсу: у кінці додано заголовок відповіді асистента
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель — діалог, відформатований шаблоном розмови.

In [ ]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Відкидаємо токени промпту: декодуємо лише те, що згенерувала модель
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
test_prompt = "Explain me, what is quantum computing?"

# Base-модель: сирий текст без шаблону
print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

# Instruct-модель: діалог, відформатований шаблоном розмови
instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 — гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [ ]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        # Нижча температура робить відповіді на арифметичні задачі стабільнішими
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` — це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




===== Qwen/Qwen3-0.6B =====
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
<think>

</think>

Hello! How can I help?<|im_end|>




Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [6]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно — найпростіше взяти його з instruct-версії тієї самої моделі.

In [7]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` — тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** — датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [8]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [9]:
# Завантажуємо лише файл однієї частини через data_files: повний датасет займає десятки гігабайт,
# а load_dataset(..., split=...) спершу перебирає файли всіх частин і працює дуже довго
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 2057 examples [00:00, 81123.08 examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі —
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки «питання» і «відповідь».
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [10]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")  # невелика вибірка для демо
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари «питання — відповідь» на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 7473/7473 [00:00<00:00, 436832.38 examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 152413.00 examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map: 100%|██████████| 100/100 [00:00<00:00, 6990.74 examples/s]


У форматі messages: {'messages': [{'role': 'system', 'content': 'You are a math tutor. Solve problems step by step.'}, {'role': 'user', 'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'}, {'role': 'assistant', 'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [11]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,  # діалог завершений: відповідь асистента вже є
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 11 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

## 0.1. Теоретичні та практичні питання (11–21)

### 11. Скільки часу зайняло виконання `02_chat_templates`? Які проблеми виникли під час виконання?
- **Час виконання на локальній машині (`NVIDIA GeForce GTX 1070`, 8 ГБ VRAM):** `74.9 с` (`1.25 хв`) для всіх токенізаторних клітинок, форматувань шаблонів, інспекції `smoltalk2` / `gsm8k` та додаткового порівняння режимів мислення `Qwen3-0.6B` (`46.3 с` з цього часу припало на генерацію 6 відповідей до 300 токенів у розділі питань).
- **Проблеми та обмеження під час виконання:**
  1. **Нестача VRAM для одночасного завантаження двох 3B-моделей (`SmolLM3-3B-Base` + `SmolLM3-3B`):** кожна з моделей у `float16` займає ~`6.15 ГБ`, тобто разом ~`12.3 ГБ` лише для ваг (плюс KV-кеш і контекст CUDA). Ноутбук розрахований на Colab **T4 (15 ГБ VRAM)** і перевищує локальні 8 ГБ VRAM GTX 1070 (~6.2 ГБ вільних під Windows WDDM), тому клітинки 5, 11 та 13 з обома 3B-моделями локально не запускалися, а натомість шаблони й токенізатори `SmolLM3-3B-Base`/`SmolLM3-3B`/`Phi-3.5`/`Qwen3-0.6B` перевірені напряму, плюс генерацію `/think` vs `/no_think` запущено наживо на `Qwen3-0.6B`.
  2. На Windows `!pip install` із клітинки 1 не запускався, щоб не пошкодити ізольоване CUDA-середовище (`torch 2.6.0+cu124` для GTX 1070 `sm_61`), а `huggingface_hub` вивів стандартне попередження про відсутність `hf_xet` та симлінків без Developer Mode.

---

### 12. Навіщо змінювати `add_generation_prompt` в `apply_chat_template()`? Як зрозуміти, яке значення параметру використано, дивлячись на результат виконання функції?
- **`add_generation_prompt=False`** використовується для формування **повного діалогу (наприклад, тренувальних прикладів SFT)**: кожна репліка вже закрита тегом кінця ходу (`<|im_end|>`, `<|end|>`), нового заголовка для генерації немає.
- **`add_generation_prompt=True`** використовується **під час інференсу**: у кінець рядка додається відкриваючий заголовок відповіді асистента (`<|im_start|>assistant\n` у `SmolLM3` або `<|im_start|>assistant\n<think>\n\n</think>\n\n` у `Qwen3` при `enable_thinking=False`), щоб модель знала, з якого місця генерувати продовження.
- **Як визначити за результатом:** у виводі клітинок 7 і 17 чітко видно: якщо рядок закінчується закриваючим тегом останнього повідомлення (`<|im_end|>`), використано `add_generation_prompt=False`; якщо наприкінці додано незакритий заголовок `<|im_start|>assistant` (або `<|im_start|>assistant\n<think>\n\n</think>\n\n`), це `add_generation_prompt=True`.

---

### 13. В чому різниця між ролями `system` та `assistant`?
- **`system`** — системний контекст керування: задає роль, правила формату (наприклад, `"Reply with JSON only"`), стиль або режим міркування (`/think` чи `/no_think`). На цих токенах модель зазвичай не штрафується функцією втрат при SFT (`labels = -100`).
- **`assistant`** — відповіді самої моделі: у багатоходовій історії це попередні репліки моделі, а під час SFT саме на токенах `assistant` обчислюється loss і навчається модель.

---

### 14. Прокоментуйте результат виконання коду в розділі 2.
З реального виводу клітинки 7 для `SmolLM3-3B` видно три важливі деталі:
1. **Автоматична ін'єкція метаданих:** навіть у `simple_qa`, де передано лише `"/no_think"`, шаблон сам вставив `Knowledge Cutoff Date: June 2025`, поточну дату (`Today Date: 11 November 2026`), `Reasoning Mode: /no_think` і дефолтну системну інструкцію `SmolLM`.
2. **Формат ChatML:** кожне повідомлення розділене токенами `<|im_start|>роль ... <|im_end|>`.
3. **Різниця `add_generation_prompt`:** при `False` діалог завершується після `<|im_end|>`, а при `True` додається `<|im_start|>assistant` як точка старту генерації. Отже, модель отримує суттєво довший службовий контекст, ніж здається за коротким списком `messages`.

---

### 15. Прокоментуйте різницю між результатами моделей Base та Instruct з розділу 3.
- **Base-модель (`SmolLM3-3B-Base`)** тренувалася лише на передбаченні наступного токена на сирому веб-корпусі: у неї навіть немає `chat_template` (це підтверджено в клітинці 19: `base_tokenizer.chat_template is None`). Отримуючи сирий рядок `"Explain me, what is quantum computing?"`, base-модель поводиться як автодоповнення тексту (генерує схожі запитання, статті чи форуми) і не знає спецтокенів кінця відповіді.
- **Instruct-модель (`SmolLM3-3B`)** навчена на діалогах із ChatML-розміткою: вона сприймає запит у `<|im_start|>user ...` саме як інструкцію помічнику, дає структуровану відповідь і коректно зупиняється на `<|im_end|>`.

---

### 16. Чому у прикладі з режимом `/think` в функцію `generate()` додатково передано `max_new_tokens=300`?
Стандартне значення `max_new_tokens=150` розраховане лише на коротку пряму відповідь. У режимі міркування модель спершу генерує внутрішній блок `<think>…</think>` із покроковою перевіркою обчислень, а лише потім — фінальну відповідь. При малому бюджеті токенів генерація обривається ще всередині міркувань (це навіть проявилося наживо у нашому тесті нижче при `300` токенах!). Тому ліміт збільшили вдвічі до `300`, щоб дати простір для ланцюжка роздумів і підсумку.

---

### 17. Чи змогла модель розв'язати математичні задачі з попереднього завдання? Який режим генерації ви б обрали, якщо вам потрібна допомога з такими задачами?
Перевіримо ті самі 3 арифметичні задачі наживо на локальній GPU за допомогою гібридної моделі **`Qwen/Qwen3-0.6B`** (`enable_thinking=False` проти `enable_thinking=True` за `max_new_tokens=300, temperature=0.3`):

In [12]:
# Живий тест на локальній GPU (GTX 1070): порівняння /no_think та /think на Qwen3-0.6B
import gc

qwen_name = "Qwen/Qwen3-0.6B"
qwen_tok = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_name, dtype=torch.float16, device_map="auto")

math_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for think_mode in [False, True]:
    print(f"===== Qwen3-0.6B | enable_thinking={think_mode} =====\n")
    for i, p in enumerate(math_prompts, 1):
        msgs = [{"role": "user", "content": p}]
        prompt_str = qwen_tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=think_mode
        )
        inputs = qwen_tok(prompt_str, return_tensors="pt").to(qwen_model.device)
        with torch.no_grad():
            out = qwen_model.generate(
                **inputs, max_new_tokens=300, temperature=0.3, do_sample=True, pad_token_id=qwen_tok.eos_token_id
            )
        ans = qwen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"Задача {i}: {p}\nВідповідь:\n{ans}\n" + "-" * 50)

del qwen_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

===== Qwen3-0.6B | enable_thinking=False =====



Задача 1: What is 15 × 24? Show your work.
Відповідь:
To find the product of 15 and 24, we can use the standard multiplication method:

15 × 24  
= 15 × (20 + 4)  
= (15 × 20) + (15 × 4)  
= 300 + 60  
= 360

So, the result is **360**.
--------------------------------------------------


Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь:
We are given that:

- 2 cups of flour is needed for **12 cookies**.

We need to find out how much flour is needed for **30 cookies**.

---

### Step 1: Find the **flour per cookie**

$$
\frac{2 \text{ cups}}{12 \text{ cookies}} = \frac{1}{6} \text{ cups per cookie}
$$

---

### Step 2: Find the amount of flour for 30 cookies

$$
30 \times \frac{1}{6} = 5 \text{ cups}
$$

---

### ✅ Final Answer: **5 cups of flour**.
--------------------------------------------------


Задача 3: If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?
Відповідь:
You have $50, and you spend:

- $18.75 on lunch  
- $12.30 on a book

So, total spent = $18.75 + $12.30 = **$31.05**

Now, subtract the total spent from the original amount:

$$
50 - 31.05 = \boxed{18.95}
$$

You have **$18.95** left.
--------------------------------------------------
===== Qwen3-0.6B | enable_thinking=True =====



Задача 1: What is 15 × 24? Show your work.
Відповідь:
<think>
Okay, so I need to figure out what 15 multiplied by 24 is. Let me start by recalling how to multiply numbers. I think there are different ways to do this, like breaking it down or using the standard algorithm. Let me try both methods to see if I get the same answer.

First, maybe I can break down one of the numbers into smaller parts. For example, 15 is the same as 10 + 5, and 24 is 20 + 4. If I use the distributive property, I can multiply each part separately and then add the results together. Let me try that.

So, 10 × 24 + 5 × 24. Let's calculate each part first. 10 × 24. Hmm, 10 times 20 is 200, and 10 times 4 is 40, so adding those together gives 240. Then, 5 × 24. Let me do that. 5 times 20 is 100, and 5 times 4 is 20, so adding those gives 120. Now, adding 240 and 120 together: 240 + 120 equals 360. So, according to this method, the product is 360.

Alternatively, maybe I can use
-------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь:
<think>
Okay, let's see. The problem says that a recipe needs 2 cups of flour for 12 cookies, and I need to figure out how much flour is needed for 30 cookies. Hmm, so I need to find the ratio between the amount of flour and the number of cookies. 

First, maybe I should figure out how much flour is needed for one cookie. If 2 cups are for 12 cookies, then dividing 2 by 12 would give me the amount of flour per cookie. Let me do that calculation. 2 divided by 12... Well, 2 divided by 12 is the same as 1/6. So, 1/6 cups of flour per cookie. 

Now, if I want 30 cookies, I need to multiply the amount per cookie by the number of cookies. So, 1/6 cups times 30 cookies. Let me write that out: (1/6) * 30. 

Calculating that, 30 divided by 6... Well, 6 goes into 30 five times, right? So that would be 5 cups. 

Wait, let me check if that makes sense. If 12 cookies take 2 cups, then 30

Задача 3: If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?
Відповідь:
<think>
Okay, let's see. I need to figure out how much money I have left after spending $18.75 on lunch and $12.30 on a book. So, first, I should add those two amounts together. Let me write that down.

Alright, so the total amount spent is $18.75 plus $12.30. Let me add those two numbers. Let me do the addition step by step. 

Starting with the cents part. 75 cents plus 30 cents. That should be 105 cents. Converting that back to dollars, 105 cents is $1.05. 

Now, adding the dollar amounts. The dollar part is $18.75 plus $12.30. Let me break that down. 

$18.75 can be thought of as $18 and 75 cents. Adding $12.30 would be adding 12 dollars and 30 cents. So, adding those together:

$18 + $12 is $30. Then, 75 cents + 30 cents is $1.05. So, total spent is $30 + $1.05, which is $31.05.

Wait, let me check that again. Maybe I should add them in another way. Let's do the additio

**Аналіз отриманого живого тесту (Питання 17):**
- **У режимі `enable_thinking=False` (`/no_think`):** компактна `Qwen3-0.6B` розв'язала **всі 3 задачі абсолютно правильно** (`360`, `5 cups`, `$18.95`) і вклалася в компактну покрокову відповідь до ~120 токенів!
- **У режимі `enable_thinking=True` (`/think`) з `max_new_tokens=300`:** модель почала ретельно розписувати альтернативні способи та самоперевірки всередині `<think>...</think>` (*«Wait, let me check that again...»*) — і в результаті **на всіх трьох задачах вичерпала ліміт у `300` токенів ще всередині блоку `<think>`**, навіть не дійшовши до фінальної відповіді користувачеві! (Хоча у проміжних міркуваннях вона вже знайшла правильні проміжні числа `360`, `5 cups` та `$31.05`).
- **Який режим обрати:**
  - Для **складних багатоетапних математичних, логічних чи алгоритмічних задач** варто обирати режим **`/think` (`enable_thinking=True`)**, бо самоперевірка знижує ризик помилок на складних кроках — але **обов'язково з достатнім `max_new_tokens` (512–2048+)**, інакше відповідь обірветься в `<think>`.
  - Для простих арифметичних запитань при жорсткому обмеженні токенів чи затримки краще спрацьовує `/no_think` із покроковим промптом або великим запасом токенів.

---

### 18. Які відмінності в шаблонах розмови різних моделей ви помітили в розділі 5?
За реальним виводом клітинки 15 видно три різні формати для тих самих ролей `system / user / assistant`:
1. **`HuggingFaceTB/SmolLM3-3B`:** ChatML (`<|im_start|>role\n...<|im_end|>`) + автоматичне додавання блоку `Metadata` (`Knowledge Cutoff Date`, `Today Date`, `Reasoning Mode: /think`) і `## Custom Instructions`.
2. **`Qwen/Qwen3-0.6B`:** також ChatML (`<|im_start|>...<|im_end|>`), але без авто-метаданих дати; прапорець `enable_thinking=False` додає порожні теги `<think>\n\n</think>\n\n` перед відповіддю асистента (тоді як `enable_thinking=True` у клітинці 17 залишає місце після `<|im_start|>assistant\n` для початку міркувань).
3. **`microsoft/Phi-3.5-mini-instruct`:** зовсім інші спецтокени — **`<|system|>`, `<|user|>`, `<|assistant|>`** і єдиний закриваючий тег **`<|end|>`** (плюс `<|endoftext|>` наприкінці).
4. **Base проти Instruct (клітинка 19):** у base-токенізатора шаблон відсутній (`False`), в instruct — є (`True`).

---

### 19. Чому нам потрібно знати про існування шаблонів розмови при роботі з SFT?
1. **Узгодженість тренування й інференсу:** модель вчиться сприймати ролі та зупинятися виключно за спецтокенами свого шаблону (`<|im_end|>`, `<|end|>`). Якщо в SFT і на інференсі шаблони не збігаються, модель ламає діалог або не зупиняє генерацію.
2. **Узгодженість режимів міркувань:** як показала клітинка 27 (`gsm8k`), якщо шаблон додає `/think`, а в тренувальних відповідях немає тегів `<think>…</think>`, модель почне пропускати міркування навіть у режимі `/think`.
3. **Точне маскування втрат (`assistant_only_loss`):** під час SFT лос зазвичай рахується лише на токенах асистента — бібліотека TRL визначає їх межі саме через `chat_template` (і теги `{% generation %}`).
4. **Робота з Base-моделями:** перед SFT для base-моделі необхідно перенести `chat_template` з instruct-версії (`clone_chat_template`).

---

### 20. На основі яких даних створено датасет `HuggingFaceTB/smoltalk2`?
Як показала інспекція клітинок 21–22, конфіг `SFT` датасету `HuggingFaceTB/smoltalk2` складається з **25 спеціалізованих частин (splits)** у режимах `_think` і `_no_think`:
- **Синтетичні діалоги та міркування від сильніших моделей-вчителів** (передусім `Qwen3-32B`, наприклад перевірений нами `smoltalk_everyday_convs_reasoning_Qwen3_32B_think` на 2 057 прикладів із розгорнутим `<think>...</think>`, а також `OpenThoughts3_1.2M`, `s1k_1.1`, `Mixture_of_Thoughts_science`);
- **Спеціалізовані прикладні піднабори:** довгий контекст (`LongAlign_64k`), багатомовні діалоги (`smoltalk_multilingual8`, `aya_dataset`), виклик інструментів (`smolagents_toolcalling_traces`, `hermes_function_calling_v1`, `xlam_traces`), табличні задачі (`table_gpt`), слідування інструкціям (`tulu_3_sft_personas_instruction_following`) та діалоги стилю SmolLM3 (`smol_magpie_ultra`, `smol_rewrite`, `smol_summarize`, `everyday_conversations`).

---

### 21. Які модифікації даних відбуваються в частині «Перетворення власного датасету у формат messages»?
1. Початковий датасет `openai/gsm8k` має плоский табличний формат із двома рядковими колонками: `question` і `answer`.
2. Функція `to_messages()` через `.map(..., batched=True, remove_columns=gsm8k.column_names)` **перетворює кожну пару на стандартний розмовний список `messages` із трьома ролями**:
   - `system`: спільна інструкція `"You are a math tutor. Solve problems step by step."`;
   - `user`: текст задачі (`question`);
   - `assistant`: еталонний покроковий розв'язок із фінальною відповіддю (`#### 72`);
   - та видаляє старі колонки `question` і `answer`.
3. У клітинці 27 до `gsm8k_messages[0]["messages"]` застосовано `instruct_tokenizer.apply_chat_template(..., add_generation_prompt=False)`, що сформувало фінальний рядок довжиною **`145` токенів** — із системним блоком метаданих `/think` на початку й відповіддю без `<think>` наприкінці (демонструючи важливість узгодження шаблону й даних).